# Two-layer FFN on PYNQ-Z2

Run one cell at a time using the assigned PYNQ-Z2. Start with B11. Read README.md before programming and stop on any error. A Python timeout cannot interrupt a blocked MMIO access.


## 1. Read-only preflight
No PYNQ import or hardware access. Run this from the extracted package directory. Use a fresh kernel; after a previous session, restart only following a successful `session.close()`.


In [3]:
from pathlib import Path
import sys
package = Path.cwd().resolve()
sys.path.insert(0, str(package))
from ffn_host import FFNSession
session = FFNSession(package)
print(session.manifest['config'])
print('PREFLIGHT PASS; physical board not accessed')


{'config_id': 'K44', 'P': '4', 'Q': '4', 'compute_pipeline': 'on', 'requested_ii': '1', 'weight_layout': 'packed', 'logical_banks_per_layer': '4', 'word_bits': '64', 'storage_type': 'ram_1p', 'storage_impl': 'bram', 'part': 'xc7z020clg400-1', 'clock_ns': '10', 'planned_rtl_validation': 'yes', 'planned_board_role': 'candidate'}
PREFLIGHT PASS; physical board not accessed


## 2. Program and check clock
The cooperative lock remains held until a normal close. This cell does not read FFN registers. Wait for `programmed` at 100 MHz before proceeding.


In [4]:
session.program()


{'clock_mhz': 100.0, 'stage': 'programmed'}

## 3. First PL access and unloaded-state checks
If this cell hangs, stop and arrange board recovery; do not rerun download or restart the kernel. Success returns `guards: pass`.


In [5]:
session.check_control()


{'guards': 'pass', 'stage': 'control'}

## 4. Allocate and clean buffers
No inference has been launched. Verify stage `allocated`.


In [6]:
session.allocate_buffers()


{'addresses': ['0x18050000', '0x18049000', '0x1804a000'], 'stage': 'allocated'}

## 5. One known input
Success requires all 64 outputs to match.


In [7]:
session.smoke()


{'correct': True, 'elapsed_ns': 1127114}

## 6. Complete correctness suite
Nine parameter sets, 206 inputs, including signed routing, ReLU, rounding, saturation, persistence and replacement.


In [8]:
validation = session.validate()
print('Validated inferences:', sum(r['inferences'] for r in validation))
validation


Validated inferences: 206


[{'case': 'random_506', 'correct': True, 'inferences': 64, 'load_ns': 2694326},
 {'case': 'random_507', 'correct': True, 'inferences': 64, 'load_ns': 1237449},
 {'case': 'random_508', 'correct': True, 'inferences': 64, 'load_ns': 1152975},
 {'case': 'zero', 'correct': True, 'inferences': 3, 'load_ns': 1144622},
 {'case': 'bias_only', 'correct': True, 'inferences': 1, 'load_ns': 1156234},
 {'case': 'routing', 'correct': True, 'inferences': 4, 'load_ns': 1139458},
 {'case': 'extremes', 'correct': True, 'inferences': 2, 'load_ns': 1163138},
 {'case': 'cancellation',
  'correct': True,
  'inferences': 2,
  'load_ns': 1152025},
 {'case': 'rounding', 'correct': True, 'inferences': 2, 'load_ns': 1141108}]

## 7. Measure FPGA host latency
20 warm-ups and 3 × 1000 calls. Input copy/flush, control, polling and output invalidate/copy are included; comparison and file I/O are excluded.


In [9]:
session.benchmark()


{'boundary': 'input copy/flush, control MMIO/poll, output invalidate/copy; no validation or CSV I/O',
 'calls': 3000,
 'correct': True,
 'load_ns': 1868545,
 'median_ns': 964715.0,
 'metadata': {'bit_sha256': '69f95be81171161e73e6ddbd4676ec1fbe5ba24dba0a8c0ea7a3c4624b128760',
  'board_model': 'unknown',
  'config': {'P': '4',
   'Q': '4',
   'clock_ns': '10',
   'compute_pipeline': 'on',
   'config_id': 'K44',
   'logical_banks_per_layer': '4',
   'part': 'xc7z020clg400-1',
   'planned_board_role': 'candidate',
   'planned_rtl_validation': 'yes',
   'requested_ii': '1',
   'storage_impl': 'bram',
   'storage_type': 'ram_1p',
   'weight_layout': 'packed',
   'word_bits': '64'},
  'data_manifest': {'cases': [{'inferences': 64, 'name': 'random_506'},
    {'inferences': 64, 'name': 'random_507'},
    {'inferences': 64, 'name': 'random_508'},
    {'inferences': 3, 'name': 'zero'},
    {'inferences': 1, 'name': 'bias_only'},
    {'inferences': 4, 'name': 'routing'},
    {'inferences': 2, 'na

## 8. Normal cleanup
Run only after successful operation. It deliberately refuses to free buffers after uncertain MMIO, timeout or mismatch.


In [11]:
session.close()
print('CLOSED: buffers released; another package may now be tested')


CLOSED: buffers released; another package may now be tested


## 9. Board CPU reference
This compiles the included C++ source on ARM, validates both layers and records CPU samples. Do not run it on the build machine as a board result.


In [12]:
import subprocess
subprocess.run([sys.executable, 'run_cpu.py'], cwd=str(package), check=True)


CompletedProcess(args=['/usr/bin/python3', 'run_cpu.py'], returncode=0)

## 10. Save evidence
Download these files with the package manifest. Repeat for K44 only after B11 has passed and closed normally. CPU compute time and Python/FPGA end-to-end time have different boundaries; report both.


In [13]:
for name in ['manifest.json', 'load_stages.jsonl', 'board_validation.json', 'board_raw.csv', 'board_results.json', 'cpu_run.log', 'cpu_raw.csv', 'cpu_results.json']:
    p = package / name
    print(name, p.stat().st_size if p.exists() else 'MISSING')


manifest.json 7357
load_stages.jsonl 1763
board_validation.json 6673
board_raw.csv 57137
board_results.json 5956
cpu_run.log 87
cpu_raw.csv 53237
cpu_results.json 6201
